# G-Code Embedding Fine-Tuning: Gun Part Classifier

Hybrid approach: statistical features from g-code + text embeddings from sampled g-code lines.

**Data layout:**
- `../data-prep/g-code-spliced/` — g-code files (gun parts, label=1)
- `../data-prep/stl-manifest.json` — metadata/labels for the above
- `../data-prep/g-code-non-gun/` — (future) non-gun g-code files (label=0)

In [ ]:
import subprocess, sys, os

VENV_DIR = os.path.join(os.getcwd(), 'venv')
VENV_PYTHON = os.path.join(VENV_DIR, 'Scripts', 'python.exe') if os.name == 'nt' else os.path.join(VENV_DIR, 'bin', 'python')

if not os.path.exists(VENV_DIR):
    print('Creating venv...')
    subprocess.check_call([sys.executable, '-m', 'venv', VENV_DIR])
    print('Installing packages...')
    subprocess.check_call([VENV_PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
    subprocess.check_call([VENV_PYTHON, '-m', 'pip', 'install',
        'torch', 'transformers', 'peft', 'accelerate', 'sentence-transformers',
        'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn',
        'datasets', 'huggingface_hub', 'tqdm', 'ipykernel'])
    # Register the venv as a Jupyter kernel
    subprocess.check_call([VENV_PYTHON, '-m', 'ipykernel', 'install', '--user', '--name', 'gcode-finetune', '--display-name', 'G-Code Finetune'])
    print('\n=== venv created. Switch to the "G-Code Finetune" kernel, then re-run from the next cell. ===')
else:
    print(f'venv already exists at {VENV_DIR}')
    print('Make sure you are using the "G-Code Finetune" kernel.')

In [ ]:
import random
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Configuration

In [ ]:
HF_DATASET = "jungter/gcode-to-model-gn"

EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
TEXT_SAMPLE_LINES = 2048
MAX_TEXT_LENGTH = 384

BATCH_SIZE = 16
LEARNING_RATE = 2e-5
LORA_LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.05
EPOCHS = 8
EARLY_STOP_PATIENCE = 3
VAL_SPLIT = 0.2
RANDOM_SEED = 42

LORA_R = 4
LORA_ALPHA = 8
LORA_DROPOUT = 0.25
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

LABEL_SMOOTHING = 0.1  # soft labels: 0 -> 0.05, 1 -> 0.95
AUGMENT_DROP_RATE = 0.15  # randomly drop 15% of g-code lines during training
CONFIDENCE_PENALTY_WEIGHT = 0.1  # penalise over-confident predictions

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 2. Load Dataset from HuggingFace

In [ ]:
from datasets import load_dataset

hf_ds = load_dataset(HF_DATASET, split="train")
print(f"Loaded {len(hf_ds)} samples from {HF_DATASET}")
print(hf_ds)

df = pd.DataFrame({
    "gcode_text": hf_ds["gcode_text"],
    "label": hf_ds["label"],
    "category": hf_ds["category"],
    "part": hf_ds["part"],
    "filename": hf_ds["filename"],
})

print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nCategories: {df['category'].unique()}")

## 3. G-Code Feature Extraction

Parse each g-code file into a fixed-length numerical feature vector capturing:
- Movement statistics (G0 vs G1 counts, travel vs extrusion ratio, layer count)
- Coordinate statistics computed from actual movements (X/Y/Z ranges, centroids, spread)
- Speed/feed rate statistics
- Extrusion statistics (total, rate distribution)

In [ ]:
def extract_gcode_features(gcode_text: str) -> dict:
    """Extract numerical features from g-code movement commands only (no header metadata)."""
    lines = gcode_text.splitlines()
    total_lines = len(lines)

    # Parse movement commands only — skip header comments entirely
    # to avoid data leakage from identical slicer boilerplate.
    g0_count = 0
    g1_count = 0
    x_vals, y_vals, z_vals = [], [], []
    e_vals = []
    f_vals = []
    layer_count = 0

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith(";"):
            if ";LAYER:" in stripped.upper():
                layer_count += 1
            continue

        if stripped.startswith("G0 ") or stripped.startswith("G0\t"):
            g0_count += 1
        elif stripped.startswith("G1 ") or stripped.startswith("G1\t"):
            g1_count += 1

        parts = stripped.split(";")[0].split()
        for p in parts:
            try:
                if p.startswith("X"): x_vals.append(float(p[1:]))
                elif p.startswith("Y"): y_vals.append(float(p[1:]))
                elif p.startswith("Z"): z_vals.append(float(p[1:]))
                elif p.startswith("E"): e_vals.append(float(p[1:]))
                elif p.startswith("F"): f_vals.append(float(p[1:]))
            except ValueError:
                pass

    def safe_stats(vals):
        if not vals:
            return 0, 0, 0, 0
        arr = np.array(vals)
        return float(arr.min()), float(arr.max()), float(arr.mean()), float(arr.std())

    x_min, x_max, x_mean, x_std = safe_stats(x_vals)
    y_min, y_max, y_mean, y_std = safe_stats(y_vals)
    z_min, z_max, z_mean, z_std = safe_stats(z_vals)
    e_min, e_max, e_mean, e_std = safe_stats(e_vals)
    f_min, f_max, f_mean, f_std = safe_stats(f_vals)

    total_moves = g0_count + g1_count
    extrusion_ratio = g1_count / max(total_moves, 1)

    features = {
        # File stats
        "total_lines": total_lines,
        "layer_count": layer_count,
        "g0_count": g0_count,
        "g1_count": g1_count,
        "total_moves": total_moves,
        "extrusion_ratio": extrusion_ratio,
        # Coordinate stats (computed from actual movements)
        "x_min": x_min, "x_max": x_max, "x_mean": x_mean, "x_std": x_std,
        "x_range": x_max - x_min,
        "y_min": y_min, "y_max": y_max, "y_mean": y_mean, "y_std": y_std,
        "y_range": y_max - y_min,
        "z_min": z_min, "z_max": z_max, "z_mean": z_mean, "z_std": z_std,
        "z_range": z_max - z_min,
        # Extrusion stats
        "e_min": e_min, "e_max": e_max, "e_mean": e_mean, "e_std": e_std,
        # Feed rate stats
        "f_min": f_min, "f_max": f_max, "f_mean": f_mean, "f_std": f_std,
    }
    return features

# Test on first sample
test_features = extract_gcode_features(df["gcode_text"].iloc[0])
print(f"Feature count: {len(test_features)}")
for k, v in test_features.items():
    print(f"  {k}: {v:.4f}")

## 4. G-Code Text Sampling

Sample lines from the beginning, middle, and end of each g-code file to capture
the overall structure for the text embedding branch.

In [ ]:
def sample_gcode_text(gcode_text: str, n_lines: int = TEXT_SAMPLE_LINES) -> str:
    """Sample lines from g-code text, stripping slicer header/boilerplate."""
    raw_lines = [l.strip() for l in gcode_text.splitlines() if l.strip()]

    # Strip header comments and slicer boilerplate that are identical across files.
    # Keep only actual movement commands (G0/G1), type markers, and layer markers.
    lines = []
    for l in raw_lines:
        if l.startswith(";"):
            upper = l.upper()
            # Keep layer/type markers -- they encode structure
            if any(k in upper for k in [";LAYER:", ";TYPE:", ";MESH:"]):
                lines.append(l)
            # Skip all other comments (slicer header, metadata, timestamps)
            continue
        lines.append(l)

    if len(lines) <= n_lines:
        return "\n".join(lines)

    third = n_lines // 3
    remainder = n_lines - 2 * third

    begin = lines[:third]
    mid_start = (len(lines) - third) // 2
    middle = lines[mid_start:mid_start + third]
    end = lines[-remainder:]

    sampled = begin + ["\n;--- MID SAMPLE ---"] + middle + ["\n;--- END SAMPLE ---"] + end
    return "\n".join(sampled)


def augment_gcode_text(gcode_text: str, drop_rate: float = AUGMENT_DROP_RATE) -> str:
    """Randomly drop lines from g-code for training-time augmentation.
    
    This prevents the model from memorising exact g-code sequences
    by giving it a different view of the same file each epoch.
    """
    lines = gcode_text.splitlines()
    if drop_rate <= 0 or len(lines) < 20:
        return gcode_text
    kept = [l for l in lines if random.random() > drop_rate]
    # Ensure we always keep at least 50% of lines
    if len(kept) < len(lines) * 0.5:
        kept = random.sample(lines, max(len(lines) // 2, 10))
    return "\n".join(kept)


# Test
sample_text = sample_gcode_text(df["gcode_text"].iloc[0])
print(f"Sampled text length: {len(sample_text)} chars")
print(sample_text[:500])


## 5. Extract Features, Sample Text, and Prepare Tokenizer

In [ ]:
from tqdm import tqdm

print("Extracting g-code features...")
all_features = []
all_texts = []
valid_indices = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        feats = extract_gcode_features(row["gcode_text"])
        text = sample_gcode_text(row["gcode_text"])
        all_features.append(feats)
        all_texts.append(text)
        valid_indices.append(idx)
    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")

df = df.loc[valid_indices].reset_index(drop=True)
feature_df = pd.DataFrame(all_features)
print(f"\nExtracted features for {len(feature_df)} files")
print(f"Feature columns ({len(feature_df.columns)}): {list(feature_df.columns)}")

In [ ]:
print(f"Loading tokenizer: {EMBEDDING_MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

all_texts = np.array(all_texts, dtype=object)
print(f"Prepared {len(all_texts)} sampled g-code texts")

## 6. Prepare Dataset

In [ ]:
# Normalize numerical features
scaler = StandardScaler()
feature_matrix = scaler.fit_transform(feature_df.values.astype(np.float32))
# Replace any NaN/inf from bad g-codes
feature_matrix = np.nan_to_num(feature_matrix, nan=0.0, posinf=0.0, neginf=0.0)

labels = df["label"].values

print(f"Feature matrix: {feature_matrix.shape}")
print(f"Sampled texts: {len(all_texts)}")
print(f"Labels: {labels.shape}, distribution: {np.bincount(labels)}")

In [ ]:
# Train/val split -- group by "part" so variants of the same 3D model
# never appear in both train and val (prevents data leakage).
groups = df["part"].values
gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SPLIT, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(feature_matrix, labels, groups))

feat_train, feat_val = feature_matrix[train_idx], feature_matrix[val_idx]
text_train, text_val = all_texts[train_idx], all_texts[val_idx]
y_train, y_val = labels[train_idx], labels[val_idx]

# Verify no group leakage
train_parts = set(groups[train_idx])
val_parts = set(groups[val_idx])
overlap = train_parts & val_parts
print(f"Group split: {len(train_parts)} train parts, {len(val_parts)} val parts, {len(overlap)} overlap")
if overlap:
    print(f"  WARNING: overlapping parts: {overlap}")

print(f"Train: {len(y_train)} (pos={y_train.sum()}, neg={len(y_train)-y_train.sum()})")
print(f"Val:   {len(y_val)} (pos={y_val.sum()}, neg={len(y_val)-y_val.sum()})")


In [ ]:
def batch_tokenize(texts, tokenizer, max_length):
    enc = tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    return enc["input_ids"], enc["attention_mask"]

print("Tokenizing val text (fixed)...")
val_input_ids, val_attention_mask = batch_tokenize(text_val, tokenizer, MAX_TEXT_LENGTH)


class GCodeDataset(Dataset):
    """Dataset with optional on-the-fly text augmentation for training."""

    def __init__(self, features, texts, labels, tokenizer, max_length, augment=False):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.texts = texts  # raw text strings
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = self.texts[idx]
        if self.augment:
            text = augment_gcode_text(text, AUGMENT_DROP_RATE)
        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return (
            self.features[idx],
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            self.labels[idx],
        )


class GCodeValDataset(Dataset):
    """Val dataset with pre-tokenized text (no augmentation)."""

    def __init__(self, features, input_ids, attention_mask, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.features[idx],
            self.input_ids[idx],
            self.attention_mask[idx],
            self.labels[idx],
        )


# Training dataset uses on-the-fly augmentation (re-tokenizes each epoch)
train_ds = GCodeDataset(feat_train, text_train, y_train, tokenizer, MAX_TEXT_LENGTH, augment=True)
val_ds = GCodeValDataset(feat_val, val_input_ids, val_attention_mask, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


## 7. Hybrid Classifier + LoRA Text Encoder

Two branches:
- **Feature branch**: MLP on extracted g-code statistics
- **Text branch**: mean-pooled encoder output from a **LoRA-adapted** embedding model

Concatenated, then classification head.

In [ ]:
class HybridGCodeClassifier(nn.Module):
    def __init__(self, n_features: int, text_encoder: nn.Module, text_hidden_dim: int, hidden_dim: int = 96):
        super().__init__()
        self.text_encoder = text_encoder

        # Feature branch -- with BatchNorm and higher dropout
        self.feature_branch = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.50),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )

        # Text branch -- with BatchNorm and higher dropout
        self.text_branch = nn.Sequential(
            nn.Linear(text_hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.50),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.50),
            nn.Linear(hidden_dim // 2, 1),
        )

    def masked_mean_pool(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

    def forward(self, features, input_ids, attention_mask):
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled_text = self.masked_mean_pool(text_out.last_hidden_state, attention_mask)

        feat_out = self.feature_branch(features)
        txt_out = self.text_branch(pooled_text)
        combined = torch.cat([feat_out, txt_out], dim=1)
        return self.classifier(combined).squeeze(-1)

    def get_embedding(self, features, input_ids, attention_mask):
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled_text = self.masked_mean_pool(text_out.last_hidden_state, attention_mask)
        feat_out = self.feature_branch(features)
        txt_out = self.text_branch(pooled_text)
        return torch.cat([feat_out, txt_out], dim=1)


print(f"Loading base text encoder: {EMBEDDING_MODEL_NAME}")
base_text_encoder = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME, trust_remote_code=True)
base_text_encoder.to(DEVICE)

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)

text_encoder = get_peft_model(base_text_encoder, lora_config)
text_encoder.to(DEVICE)
text_encoder.print_trainable_parameters()

n_features = feature_matrix.shape[1]
text_hidden_dim = base_text_encoder.config.hidden_size
model = HybridGCodeClassifier(n_features, text_encoder, text_hidden_dim).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


## 8. Training Loop

In [ ]:
# Handle class imbalance with weighted loss
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
if n_neg > 0 and n_pos > 0:
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
else:
    pos_weight = torch.tensor([1.0], dtype=torch.float32).to(DEVICE)
    print("WARNING: Missing one class in train split. Metrics will be unstable.")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


def compute_confidence_penalty(logits):
    """Entropy-based penalty discouraging over-confident predictions.
    
    Maximises entropy so the model is less likely to output 0.99/0.01
    on everything. This is key to fixing the 99%-confidence-on-unseen problem.
    """
    probs = torch.sigmoid(logits)
    probs = probs.clamp(1e-6, 1 - 1e-6)
    entropy = -(probs * probs.log() + (1 - probs) * (1 - probs).log())
    return -entropy.mean()  # negative because we WANT higher entropy


head_params = []
lora_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if "text_encoder" in name:
        lora_params.append(param)
    else:
        head_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": head_params, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
        {"params": lora_params, "lr": LORA_LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    ]
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1
)


In [ ]:
train_losses = []
val_losses = []
val_accs = []
best_val_loss = float("inf")
best_epoch = -1
patience_counter = 0

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_loss = 0.0
    for feat_batch, input_ids, attention_mask, label_batch in train_loader:
        feat_batch = feat_batch.to(DEVICE)
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        label_batch = label_batch.to(DEVICE)

        # Apply label smoothing: 0 -> 0.05, 1 -> 0.95
        smooth_labels = label_batch * (1 - LABEL_SMOOTHING) + LABEL_SMOOTHING / 2

        optimizer.zero_grad()
        logits = model(feat_batch, input_ids, attention_mask)
        bce_loss = criterion(logits, smooth_labels)

        # Confidence penalty: discourage over-certain predictions
        conf_penalty = compute_confidence_penalty(logits)
        loss = bce_loss + CONFIDENCE_PENALTY_WEIGHT * conf_penalty

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(label_batch)

    train_loss = epoch_loss / len(train_ds)
    train_losses.append(train_loss)

    # Validate
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for feat_batch, input_ids, attention_mask, label_batch in val_loader:
            feat_batch = feat_batch.to(DEVICE)
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            label_batch = label_batch.to(DEVICE)

            logits = model(feat_batch, input_ids, attention_mask)
            loss = criterion(logits, label_batch)  # val uses hard labels
            val_loss += loss.item() * len(label_batch)

            preds = (torch.sigmoid(logits) > 0.5).int()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label_batch.int().cpu().numpy())

    val_loss /= len(val_ds)
    val_losses.append(val_loss)
    val_acc = np.mean(np.array(all_preds) == np.array(all_labels))
    val_accs.append(val_acc)

    scheduler.step(val_loss)

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save(model.state_dict(), "best_hybrid_classifier.pt")
    else:
        patience_counter += 1

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f}")

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}. Best epoch: {best_epoch}")
        break


## 9. Evaluation

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(train_losses, label="Train")
axes[0].plot(val_losses, label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(val_accs, label="Val Acc", color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].set_title("Validation Accuracy")

# Loss gap (overfitting indicator)
gap = [v - t for t, v in zip(train_losses, val_losses)]
axes[2].plot(gap, label="Val - Train Loss", color="red")
axes[2].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss Gap")
axes[2].legend()
axes[2].set_title("Overfitting Gap")

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
print('Saved training_curves.png')
plt.close()


In [ ]:
# Load best model and evaluate
model.load_state_dict(torch.load("best_hybrid_classifier.pt", map_location=DEVICE))
model.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for feat_batch, input_ids, attention_mask, label_batch in val_loader:
        feat_batch = feat_batch.to(DEVICE)
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(feat_batch, input_ids, attention_mask)
        probs = torch.sigmoid(logits)
        preds = probs > 0.5

        all_preds.extend(preds.int().cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(label_batch.int().numpy())

all_preds = np.array(all_preds, dtype=int)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels, dtype=int)

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["non-gun", "gun"]))

if len(np.unique(all_labels)) > 1:
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["non-gun", "gun"], yticklabels=["non-gun", "gun"], ax=axes[0])
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    axes[0].set_title("Confusion Matrix")

    # Confidence distribution histogram
    gun_probs = all_probs[all_labels == 1]
    nongun_probs = all_probs[all_labels == 0]
    axes[1].hist(gun_probs, bins=30, alpha=0.6, label='Gun (actual)', color='red')
    axes[1].hist(nongun_probs, bins=30, alpha=0.6, label='Non-gun (actual)', color='blue')
    axes[1].set_xlabel('Predicted probability (gun)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Confidence Distribution')
    axes[1].legend()
    axes[1].axvline(x=0.5, color='k', linestyle='--', alpha=0.3)

    # Calibration plot
    from sklearn.calibration import calibration_curve
    fraction_pos, mean_predicted = calibration_curve(all_labels, all_probs, n_bins=10, strategy='uniform')
    axes[2].plot(mean_predicted, fraction_pos, 's-', label='Model', color='blue')
    axes[2].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    axes[2].set_xlabel('Mean predicted probability')
    axes[2].set_ylabel('Fraction of positives')
    axes[2].set_title('Calibration Curve')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig('evaluation_plots.png', dpi=150, bbox_inches='tight')
    print('Saved evaluation_plots.png')
    plt.close()

    print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}")

    tn, fp, fn, tp = cm.ravel()
    print(f"\nTrue Positives:  {tp} (gun correctly identified)")
    print(f"True Negatives:  {tn} (non-gun correctly identified)")
    print(f"False Positives: {fp} (non-gun misclassified as gun)")
    print(f"False Negatives: {fn} (gun misclassified as non-gun)")

    # Check for overconfidence
    high_conf_wrong = ((all_probs > 0.9) & (all_labels == 0)).sum() + ((all_probs < 0.1) & (all_labels == 1)).sum()
    print(f"\nOverconfident & wrong: {high_conf_wrong} samples (prob>0.9 but non-gun, or prob<0.1 but gun)")
    if high_conf_wrong > 0:
        print("  -> Model is still overconfident. Consider more diverse non-gun data.")
else:
    print("Only one class present -- add non-gun samples to compute meaningful metrics.")


## 10. Inference Helper

Classify a new g-code file.

In [ ]:
def classify_gcode(gcode_input: str, model, tokenizer, scaler, threshold=0.5):
    """Classify g-code. Accepts raw text or a filepath. Returns (is_gun: bool, confidence: float)."""
    if os.path.isfile(gcode_input):
        with open(gcode_input, "r", errors="ignore") as f:
            gcode_text = f.read()
    else:
        gcode_text = gcode_input

    features = extract_gcode_features(gcode_text)
    feat_vec = scaler.transform(
        np.array([list(features.values())], dtype=np.float32)
    )
    feat_vec = np.nan_to_num(feat_vec, nan=0.0, posinf=0.0, neginf=0.0)

    text = sample_gcode_text(gcode_text)
    text_enc = tokenizer(
        [text],
        padding="max_length",
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors="pt",
    )

    model.eval()
    with torch.no_grad():
        feat_t = torch.tensor(feat_vec, dtype=torch.float32).to(DEVICE)
        input_ids = text_enc["input_ids"].to(DEVICE)
        attention_mask = text_enc["attention_mask"].to(DEVICE)
        logit = model(feat_t, input_ids, attention_mask)
        prob = torch.sigmoid(logit).item()

    return prob >= threshold, prob

# Test inference on a local g-code file
is_gun, confidence = classify_gcode('output.gcode', model, tokenizer, scaler)
print(f"File: output.gcode")
print(f"Prediction: {'GUN PART' if is_gun else 'NOT gun part'} (confidence: {confidence:.4f})")

## 11. Save Model & Artifacts

In [ ]:
import pickle

SAVE_DIR = Path("./saved_model")
SAVE_DIR.mkdir(exist_ok=True)

# Save full best model weights for quick reload
torch.save(model.state_dict(), SAVE_DIR / "hybrid_classifier_with_lora.pt")

# Save LoRA adapter and tokenizer
model.text_encoder.save_pretrained(SAVE_DIR / "lora_adapter")
tokenizer.save_pretrained(SAVE_DIR / "tokenizer")

# Save scaler
with open(SAVE_DIR / "feature_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save config
config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "n_features": int(n_features),
    "text_hidden_dim": int(text_hidden_dim),
    "feature_columns": list(feature_df.columns),
    "text_sample_lines": TEXT_SAMPLE_LINES,
    "max_text_length": MAX_TEXT_LENGTH,
    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
    },
}
with open(SAVE_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Model and artifacts saved to {SAVE_DIR}")
print(f"Files: {list(SAVE_DIR.iterdir())}")

## 12. Feature Importance Analysis (Optional)

Quick look at which numerical features matter most.

In [ ]:
# Visualize feature distributions (useful when you have both classes)
if len(df["label"].unique()) > 1:
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    axes = axes.flatten()
    cols = feature_df.columns[:20]
    for i, col in enumerate(cols):
        gun_vals = feature_df.loc[df["label"] == 1, col]
        non_gun_vals = feature_df.loc[df["label"] == 0, col]
        axes[i].hist(gun_vals, alpha=0.5, label="gun", bins=30)
        axes[i].hist(non_gun_vals, alpha=0.5, label="non-gun", bins=30)
        axes[i].set_title(col, fontsize=9)
        axes[i].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
    print('Saved feature_distributions.png')
    plt.close()
else:
    print("Feature distribution analysis available once you add non-gun samples.")
    print("\nCurrent feature summary (gun parts only):")
    display(feature_df.describe().T)

---

## Next Steps

1. **Add non-gun g-codes** to `../data-prep/g-code-non-gun/` (any `.gcode` files will be auto-discovered with label=0)
2. **Re-run all cells** after adding negative samples
3. **Tune LoRA + head hyperparameters** (`LORA_R`, `LORA_DROPOUT`, `LEARNING_RATE`, `LORA_LEARNING_RATE`, `EPOCHS`)
4. **Try threshold calibration** on validation set instead of fixed `0.5`
5. **Export for production** — keep `lora_adapter/`, `tokenizer/`, scaler, and `config.json` together